In [2]:
import numpy as np
import pandas as pd
import uwb_dataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

data = uwb_dataset.import_from_files()  # numpy array

# column names based on CSV format
base_cols = [
    "NLOS","RANGE","FP_IDX","FP_AMP1","FP_AMP2","FP_AMP3",
    "STDEV_NOISE","CIR_PWR","MAX_NOISE","RXPACC","CH",
    "FRAME_LEN","PREAM_LEN","BITRATE","PRFR"
]
cir_cols = [f"CIR{i}" for i in range(1016)]
cols = base_cols + cir_cols

df = pd.DataFrame(data, columns=cols)
df.head()

../dataset/uwb_dataset_part2.csv
../dataset/uwb_dataset_part3.csv
../dataset/uwb_dataset_part1.csv
../dataset/uwb_dataset_part4.csv
../dataset/uwb_dataset_part5.csv
../dataset/uwb_dataset_part7.csv
../dataset/uwb_dataset_part6.csv


,NLOS,RANGE,FP_IDX,FP_AMP1,FP_AMP2,FP_AMP3,STDEV_NOISE,CIR_PWR,MAX_NOISE,RXPACC,...,CIR1006,CIR1007,CIR1008,CIR1009,CIR1010,CIR1011,CIR1012,CIR1013,CIR1014,CIR1015
0,1.0,6.18,749.0,4889.0,13876.0,10464.0,240.0,9048.0,3668.0,1024.0,...,0.798828,0.916016,0.574219,0.270508,0.709961,0.358398,0.784180,0.799805,0.456055,0.75
1,1.0,4.54,741.0,2474.0,2002.0,1593.0,68.0,6514.0,1031.0,1024.0,...,0.282227,0.222656,0.104492,0.475586,0.479492,0.394531,0.326172,0.205078,0.099609,0.00
2,1.0,4.39,744.0,1934.0,2615.0,4114.0,52.0,2880.0,796.0,1024.0,...,0.120117,0.274414,0.471680,0.094727,0.265625,0.071289,0.122070,0.165039,0.177734,0.00
3,1.0,1.27,748.0,16031.0,17712.0,10420.0,64.0,12855.0,1529.0,323.0,...,0.523220,0.427245,0.678019,0.291022,0.696594,0.479876,0.532508,0.860681,0.984520,0.00
4,0.0,1.16,743.0,20070.0,19886.0,15727.0,76.0,11607.0,2022.0,296.0,...,0.293919,0.145270,1.209459,1.040541,0.445946,0.442568,0.344595,0.425676,0.550676,0.00


In [3]:
WINDOW_SIZE = 200
LEAD        = 10  

cir_array  = df[cir_cols].to_numpy()
fp_indices = df["FP_IDX"].to_numpy().astype(int)

windowed = np.zeros((len(df), WINDOW_SIZE))

for i in range(len(df)):
    fp_idx = fp_indices[i]
    start  = max(0, fp_idx - LEAD)
    end    = min(1016, start + WINDOW_SIZE)
    actual_size               = end - start
    windowed[i, :actual_size] = cir_array[i, start:end]

df          = df.drop(columns=cir_cols)
cir_cols    = [f"CIR_{i}" for i in range(WINDOW_SIZE)]  
windowed_df = pd.DataFrame(windowed, columns=cir_cols, index=df.index)
df          = pd.concat([df, windowed_df], axis=1)

# Verify
print(f"Windowed array shape: {windowed.shape}")
print(f"New df shape: {df.shape}")
print(f"All zero rows: {(windowed == 0).all(axis=1).sum()}")


Windowed array shape: (42000, 200)
New df shape: (42000, 215)
All zero rows: 0


Basic Cleaning/Validation

In [4]:
print("Missing values:", df.isna().sum().sum())
print("Duplicates:", df.duplicated().sum())

#drop duplicates if any
df = df.drop_duplicates()

Missing values: 0
Duplicates: 0


feature engineering

In [5]:
# FP “power” aggregate
df["FP_POWER"] = (df["FP_AMP1"]**2 + df["FP_AMP2"]**2 + df["FP_AMP3"]**2)

# Simple CIR summary stats (fast, useful)
cir = df[cir_cols].to_numpy()
df["CIR_MEAN"] = cir.mean(axis=1)
df["CIR_MAX"]  = cir.max(axis=1)
df["CIR_ENERGY"] = (cir**2).mean(axis=1)  # average energy

df[["FP_POWER","CIR_MEAN","CIR_MAX","CIR_ENERGY"]].head()

df

,NLOS,RANGE,FP_IDX,FP_AMP1,FP_AMP2,FP_AMP3,STDEV_NOISE,CIR_PWR,MAX_NOISE,RXPACC,...,CIR_194,CIR_195,CIR_196,CIR_197,CIR_198,CIR_199,FP_POWER,CIR_MEAN,CIR_MAX,CIR_ENERGY
0,1.0,6.18,749.0,4889.0,13876.0,10464.0,240.0,9048.0,3668.0,1024.0,...,0.669922,0.349609,1.093750,0.911133,0.239258,0.650391,3.259410e+08,1.328721,18.063477,7.179700
1,1.0,4.54,741.0,2474.0,2002.0,1593.0,68.0,6514.0,1031.0,1024.0,...,0.172852,0.139648,0.234375,0.185547,0.274414,0.304688,1.266633e+07,0.956118,13.740234,4.419092
2,1.0,4.39,744.0,1934.0,2615.0,4114.0,52.0,2880.0,796.0,1024.0,...,0.497070,0.126953,0.444336,0.174805,0.280273,0.106445,2.750358e+07,0.823608,7.645508,2.060393
3,1.0,1.27,748.0,16031.0,17712.0,10420.0,64.0,12855.0,1529.0,323.0,...,0.832817,0.600619,0.569659,0.563467,1.030960,0.693498,6.792843e+08,2.775217,65.857585,86.415936
4,0.0,1.16,743.0,20070.0,19886.0,15727.0,76.0,11607.0,2022.0,296.0,...,0.780405,0.817568,0.385135,0.847973,1.060811,0.513514,1.045596e+09,3.797382,71.388514,105.011061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41995,0.0,4.09,745.0,1106.0,6951.0,6283.0,88.0,10076.0,1856.0,340.0,...,0.841176,0.791176,0.285294,0.755882,1.429412,0.811765,8.901573e+07,3.989191,49.488235,62.791435
41996,0.0,2.18,744.0,14870.0,13482.0,8864.0,44.0,12604.0,781.0,416.0,...,0.110577,0.627404,0.382212,0.225962,0.298077,0.127404,4.814517e+08,2.857680,48.288462,51.167367
41997,0.0,1.25,748.0,18329.0,17722.0,10731.0,60.0,10192.0,1600.0,270.0,...,0.092593,0.351852,0.359259,0.688889,0.911111,0.470370,7.651759e+08,3.634796,69.644444,98.328866
41998,0.0,3.17,747.0,3654.0,18470.0,17737.0,64.0,12018.0,967.0,789.0,...,0.519645,0.253485,0.362484,0.205323,0.389100,0.329531,6.690938e+08,1.503035,23.038023,13.341668


Data reduction using PCA
Explained variance means how much of the CIR information is the 30 components keeping

In [ ]:
from sklearn.model_selection import train_test_split

# Separate target
y = df["NLOS"].astype(int).to_numpy()

# Features excluding raw CIR for now
non_cir_features = [
    "RANGE","FP_IDX","FP_AMP1","FP_AMP2","FP_AMP3",
    "STDEV_NOISE","CIR_PWR","MAX_NOISE","RXPACC","CH",
    "FRAME_LEN","PREAM_LEN","BITRATE","PRFR",
    "FP_POWER","CIR_MEAN","CIR_MAX","CIR_ENERGY"
]

X_non_cir = df[non_cir_features].to_numpy()
X_cir = df[cir_cols].to_numpy()

X_train_idx, X_test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42, stratify=y
)
X_cir_train     = X_cir[X_train_idx]
X_cir_test      = X_cir[X_test_idx]
X_non_cir_train = X_non_cir[X_train_idx]
X_non_cir_test  = X_non_cir[X_test_idx]
y_train = y[X_train_idx]
y_test  = y[X_test_idx]

print(f"Train size: {len(X_train_idx)} ({len(X_train_idx)/len(df)*100:.1f}%)")
print(f"Test  size: {len(X_test_idx)} ({len(X_test_idx)/len(df)*100:.1f}%)")
print(f"\nTraining balance - LOS: {(y_train==0).sum()} NLOS: {(y_train==1).sum()}")
print(f"Testing balance  - LOS: {(y_test==0).sum()}  NLOS: {(y_test==1).sum()}")


Train size: 33600 (80.0%)
Test  size: 8400 (20.0%)

Training balance - LOS: 16800 NLOS: 16800
Testing balance  - LOS: 4200  NLOS: 4200


array([[0.77887789, 0.69306931, 0.94059406, ..., 1.34653465, 0.82508251,
        0.7359736 ],
       [0.56727273, 0.72727273, 0.60363636, ..., 0.89090909, 0.17454545,
        1.01818182],
       [0.84615385, 0.81318681, 0.64835165, ..., 0.88644689, 0.88278388,
        0.56776557],
       ...,
       [0.48599671, 0.65897858, 0.5354201 , ..., 0.42009885, 0.5354201 ,
        1.35914333],
       [0.84410646, 0.37262357, 1.03041825, ..., 0.53231939, 0.79467681,
        1.10646388],
       [0.16901408, 0.29753521, 0.24119718, ..., 0.24647887, 0.34507042,
        0.45070423]], shape=(33600, 200))

In [ ]:
# Scale CIR before PCA
scaler_cir = StandardScaler()
X_cir_scaled = scaler_cir.fit_transform(X_cir)

# Choose number of PCA components (start with 20–50; you can tune)
pca = PCA(n_components=30, random_state=42)
X_cir_pca = pca.fit_transform(X_cir_scaled)

print("Explained variance (30 comps):", pca.explained_variance_ratio_.sum())

Final training matrix + scaling

In [7]:
# Combine engineered + PCA features
X = np.hstack([X_non_cir, X_cir_pca])

# Scale final feature matrix (good practice for many models)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled.shape

(42000, 48)